# MLP → CNN → 簡易 Transformer

**出典**　機械学習 4 分野の教科書 — 第 8 章 2-C Transformer と生成モデル ／ 05 実装：MLP → CNN → 簡易 Transformer

**このノートのパス**　`ml-code/08_transformer/01_mlp_cnn_transformer.ipynb`

**このノートで確かめること**
3 つの構造を並べて書き、**パディングを無視させる `src_key_padding_mask`** と
**パディングを除いた平均プーリング**という、実装でいちばん間違えやすい 2 点を確かめる。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `torch` |
| 実行時間の目安 | 約 30 秒 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q torch

In [3]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# ---------- ① MLP（最も基本）----------
mlp = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28*28, 256), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(256, 128),   nn.ReLU(),
    nn.Linear(128, 10),
)

# ---------- ② CNN（画像の構造を使う）----------
cnn = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(64*7*7, 10),
)

img = torch.randn(4, 1, 28, 28)
print("MLP 出力:", tuple(mlp(img).shape), " パラメータ数:", sum(p.numel() for p in mlp.parameters()))
print("CNN 出力:", tuple(cnn(img).shape), " パラメータ数:", sum(p.numel() for p in cnn.parameters()))

MLP 出力: (4, 10)  パラメータ数: 235146
CNN 出力: (4, 10)  パラメータ数: 50378


### 同じ課題に対する、パラメータ数の違いに注目

**CNN のほうがパラメータが少ないのに画像では強い。**重みを画像全体で使い回す（重み共有）ためで、これが「構造を使う」ということ。

## ③ 簡易 Transformer（系列分類）

In [4]:
class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2,
                 num_classes=2, max_len=512):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        # 位置エンコーディング（学習可能な版。元論文は sin/cos の固定版）
        self.pos = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.normal_(self.pos, std=0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,                 # ヘッド数。d_model が nhead で割り切れる必要がある
            dim_feedforward=d_model*4,   # FFN の中間次元。慣例で 4 倍
            dropout=0.1,
            batch_first=True,            # 入力を (バッチ, 系列, 次元) の順にする
            norm_first=True,             # LayerNorm を先に置く（学習が安定しやすい）
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers, enable_nested_tensor=False)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x, pad_mask=None):
        # x: (バッチ, 系列長) のトークン ID
        h = self.emb(x) + self.pos[:, :x.size(1)]
        # pad_mask: パディング位置を True にした (バッチ, 系列長) の bool テンソル。
        #           ここを無視させないと、詰め物に注目してしまう
        h = self.encoder(h, src_key_padding_mask=pad_mask)

        # パディングを除いた平均プーリングで文全体の表現にまとめる
        if pad_mask is not None:
            valid = (~pad_mask).unsqueeze(-1).float()
            h = (h * valid).sum(1) / valid.sum(1).clamp(min=1)
        else:
            h = h.mean(1)
        return self.head(h)


model = TinyTransformer(vocab_size=30000)
x = torch.randint(1, 30000, (8, 64))          # バッチ 8、系列長 64
pad = torch.zeros(8, 64, dtype=torch.bool)    # 今回はパディングなし
print(model(x, pad).shape)                    # torch.Size([8, 2])

torch.Size([8, 2])


### マスクが本当に効いているかを確かめる

**実際にパディングのある入力**を作り、詰め物の中身を変えても出力が変わらないことを見る。

In [5]:
torch.manual_seed(0)
B, L = 8, 64
lengths = torch.randint(8, L + 1, (B,))
xp = torch.zeros(B, L, dtype=torch.long)
for i, n in enumerate(lengths):
    xp[i, :n] = torch.randint(1, 30000, (int(n),))
pad_mask = (xp == 0)                          # True = パディング位置

xp2 = xp.clone()
for i, n in enumerate(lengths):
    xp2[i, n:] = torch.randint(1, 30000, (L - int(n),))   # 詰め物を別トークンに

model.eval()
with torch.no_grad():
    a = model(xp,  pad_mask)
    b = model(xp2, pad_mask)
    c = model(xp2, None)                      # ← マスクを渡し忘れた場合
print("マスクあり  最大差:", (a - b).abs().max().item(), "← ほぼ 0 なら正しい")
print("マスクなし  最大差:", (a - c).abs().max().item(), "← 大きい＝詰め物に引きずられている")

マスクあり  最大差: 0.0 ← ほぼ 0 なら正しい
マスクなし  最大差: 0.42263519763946533 ← 大きい＝詰め物に引きずられている


## 読み取りどころ

- **`src_key_padding_mask` は True が「無視する位置」。**逆に覚えると静かに壊れる。
- **マスクを渡すだけでは足りない。**最後の平均プーリングでも `valid` を掛けて割らないと、詰め物の出力が平均に混ざる。**この 2 か所の両方が要る。**
- **`d_model` は `nhead` で割り切れる必要がある。**128 ÷ 4 = 32 が 1 ヘッドあたりの次元。割り切れないと例外になる。
- **`norm_first=True`（Pre-LN）は学習が安定しやすい。**元論文は Post-LN だが、深くすると学習率のウォームアップが要る。

⚠️ 上の最後のセルで、**マスクなしの差が大きい**ことを確認しておくこと。これが「詰め物に注目してしまう」の正体で、精度が出ない原因として非常に多い。